
# Python for Automotive Engineers — **Dictionaries**

**Focus:** Key→value mappings for signal definitions, DTC descriptions, ECU version maps, and parsed log records.

**You will learn**
- Creating dictionaries (literals, `dict()`, `zip`)
- Accessing, adding, updating, removing items
- Membership, views (`keys`, `values`, `items`), looping patterns
- `get`, `setdefault`, `update`, `pop`, `popitem`, `clear`
- Sorting/transforming with **dict comprehensions**
- Nested dictionaries (e.g., per‑ECU configs)
- Shallow vs deep copies (caution with nested data)



## 1. Creating Dictionaries


In [ ]:

# Literal
signal_units = {"speed": "km/h", "rpm": "min^-1", "temp": "C"}

# dict() and zip()
ecus = ["PCM", "ABS", "BCM"]
versions = ["v3.4.1", "v1.9.0", "v2.2.5"]
ecu_versions = dict(zip(ecus, versions))

# fromkeys (default value shared!)
default_flags = dict.fromkeys(["ABS", "PCM", "BCM"], False)
signal_units, ecu_versions, default_flags



## 2. Accessing Values
- Bracket indexing raises `KeyError` if missing.
- `get(key, default)` avoids exceptions.


In [ ]:

print("ABS version:", ecu_versions["ABS"])        # Key exists
print("IC version via get():", ecu_versions.get("IC", "N/A"))

try:
    print(ecu_versions["IC"])  # raises KeyError
except KeyError as e:
    print("KeyError:", e)



## 3. Adding & Updating
Use assignment or `update()` to merge.


In [ ]:

# Add a new ECU
ecu_versions["IC"] = "v1.1.0"

# Bulk update
patch = {"PCM": "v3.4.2"}
ecu_versions.update(patch)

ecu_versions



## 4. Removing Items
`del`, `pop(key[, default])`, `popitem()` (LIFO for 3.7+), and `clear()`.


In [ ]:

# pop and popitem
popped = ecu_versions.pop("IC")
last_key, last_val = ecu_versions.popitem()

# restore for demo
ecu_versions[last_key] = last_val

popped, (last_key, last_val), ecu_versions



## 5. Membership & Views
- `in` checks **keys** by default
- `keys()`, `values()`, `items()` return dynamic views for iteration


In [ ]:

"PCM" in ecu_versions, "v2.2.5" in ecu_versions.values(), list(ecu_versions.items())



## 6. Looping Patterns


In [ ]:

for ecu, ver in ecu_versions.items():
    print(f"{ecu} → {ver}")



## 7. Sorting Dictionaries (by key or value)


In [ ]:

# Sort by ECU name, then by version
sorted_by_key = dict(sorted(ecu_versions.items()))
sorted_by_val = dict(sorted(ecu_versions.items(), key=lambda kv: kv[1]))
sorted_by_key, sorted_by_val



## 8. Dictionary Comprehensions
Useful for transforming parsed data.


In [ ]:

# From list of DTCs to counts
dtcs = ["P0300", "P0301", "P0300", "P0420", "P0300", "U0100"]
counts = {code: dtcs.count(code) for code in set(dtcs)}

# From CAN IDs to labeled strings
can_ids = [0x100, 0x200, 0x321]
labels = {cid: f"{cid:#05x}" for cid in can_ids}
counts, labels



## 9. Nested Dictionaries (ECU → signals → attributes)


In [ ]:

vehicle = {
    "PCM": {
        "speed": {"unit": "km/h", "value": 72},
        "rpm":   {"unit": "min^-1", "value": 2100},
    },
    "ABS": {
        "wheel_speed_fl": {"unit": "km/h", "value": 70},
    }
}
vehicle["PCM"]["rpm"]["value"], vehicle.get("TCU", {}).get("mode", "N/A")



## 10. Copy Semantics (Shallow vs Deep)
Be careful when copying nested dicts.


In [ ]:

import copy

orig = {"PCM": {"rpm": 800}}
shallow = orig.copy()
deep = copy.deepcopy(orig)

# mutate nested
orig["PCM"]["rpm"] = 900
shallow["PCM"]["rpm"], deep["PCM"]["rpm"]



## 11. Parsing Strings → Dictionaries
From a CAN log line: `"0x200,8,11 22 33 44 55 66 77 88"` → dict with fields.


In [ ]:

row = "0x200,8,11 22 33 44 55 66 77 88"
can_id_str, dlc_str, payload_str = row.split(",")
parsed = {
    "id": int(can_id_str, 16),
    "dlc": int(dlc_str),
    "data": [int(b, 16) for b in payload_str.split()]
}
parsed



## 12. Error‑Safe Reads: `get` and `setdefault`


In [ ]:

# get with default
mode = vehicle.get("TCU", {}).get("mode", "DEFAULT")

# setdefault initializes a missing nested dict key
vehicle.setdefault("TCU", {}).setdefault("mode", "ECO")
mode, vehicle["TCU"]["mode"]



## 13. Exercises
1. Create `dtc_desc` mapping codes to descriptions and fetch one with `get()`.
2. Add a new ECU/version to `ecu_versions` and then **remove** it with `pop()`.
3. Iterate `vehicle` to print `ECU: signal → value unit` for all entries.
4. Build a dict comprehension from a list of CAN IDs to label strings like `"0x321 (std)"`.
5. Demonstrate shallow copy pitfalls with a nested dict.



## 14. Challenges
- **Session Aggregator**: From multiple DTC lists, produce a dict `{code: count}` sorted by count desc.
- **Calibration Checker**: Given a list of `(name, value, min, max)`, output a dict of out‑of‑range entries.
- **Signal Rename**: Rename dict keys from `"wheel_speed_fl"` to `"wheel_speed_front_left"` safely.
- **Merge Strategy**: Merge two ECU version dicts, preferring the **higher** semver per ECU.
